In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import joblib

# =========================
# Load Dataset
# =========================
dataset = pd.read_csv('dataset/liver.csv')

# =========================
# Preprocessing
# =========================
dataset.replace('?', np.nan, inplace=True)
dataset.fillna(dataset.mean(numeric_only=True), inplace=True)

# Encode Gender
le = LabelEncoder()
dataset['Gender'] = le.fit_transform(dataset['Gender'])

# Target
dataset['Dataset'] = dataset['Dataset'].apply(lambda x: 1 if x == 1 else 0)

# Split
X_df = dataset.drop('Dataset', axis=1)
y = dataset['Dataset']

# ✅ IMPORTANT: Save column names
feature_columns = X_df.columns.tolist()

# Scale
scaler = StandardScaler()
X = scaler.fit_transform(X_df)

# Train model (use RF directly — stable)
model = RandomForestClassifier(n_estimators=100)
model.fit(X, y)

# =========================
# SAVE FILES (FINAL)
# =========================
joblib.dump(model, 'liver_model.pkl')
joblib.dump(scaler, 'liver_scaler.pkl')
joblib.dump(feature_columns, 'liver_features.pkl')

print("✅ Liver model saved PERFECTLY!")

# =========================
# Prediction Function (FINAL)
# =========================
def predict_liver(new_data):
    model = joblib.load('liver_model.pkl')
    scaler = joblib.load('liver_scaler.pkl')
    cols = joblib.load('liver_features.pkl')

    df = pd.DataFrame([new_data])

    # Ensure all columns exist
    for col in cols:
        if col not in df.columns:
            df[col] = 0

    df = df[cols]

    df = scaler.transform(df)

    prob = model.predict_proba(df)[0][1] * 100

    risk = (
        "HIGH RISK" if prob >= 70 else
        "MEDIUM RISK" if prob >= 30 else
        "LOW RISK"
    )

    print(f"\n🎯 Liver Disease Probability: {prob:.2f}% | Risk: {risk}")

# =========================
# Example Test
# =========================
predict_liver({
    'Age': 60, 'Gender': 1, 'Total_Bilirubin': 4.5,
    'Direct_Bilirubin': 2.2, 'Alkaline_Phosphotase': 420,
    'Alamine_Aminotransferase': 52, 'Aspartate_Aminotransferase': 75,
    'Total_Protiens': 5.9, 'Albumin': 2.8,
    'Albumin_and_Globulin_Ratio': 0.7
})

✅ Liver model saved PERFECTLY!

🎯 Liver Disease Probability: 98.00% | Risk: HIGH RISK
